In [2]:
import asyncio
import random

async def fake_llm(prompt):
    latency = random.uniform(0.2, 1.0)
    await asyncio.sleep(latency)
    if random.random() < 0.2:                 # 20% fail
        raise ConnectionError("failed")
    return latency

async def safe_call(prompt):
    try:
        return {"ok": True, "latency": await fake_llm(prompt)}
    except Exception:
        return {"ok": False, "latency": 0}

async def process_in_batches(prompts, n):
    for i in range(0, len(prompts), n):
        batch = prompts[i:i+n]
        results = await asyncio.gather(*[safe_call(p) for p in batch])
        ok = [r for r in results if r["ok"]]
        avg = sum(r["latency"] for r in ok) / len(ok) if ok else 0
        print(f"Batch {i//n + 1}: {len(ok)}/{len(batch)} ok, avg {avg:.2f}s")

prompts = [f"prompt {i}" for i in range(10)]

# In Jupyter, you can await directly at the top level
await process_in_batches(prompts, 4)

Batch 1: 3/4 ok, avg 0.80s
Batch 2: 3/4 ok, avg 0.95s
Batch 3: 2/2 ok, avg 0.65s
